## Setup

In [4]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets

In [5]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
# from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
# from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [6]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### wikiann

In [7]:
wikiann_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

wikiann = ner.ReadNERData()
wikiann_words, wikiann_labels = wikiann.read_dataset(
    'wikiann',
    wikiann_label_map,
    lang='sk'
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

In [8]:
print(ner.check_labels(wikiann_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

{'O', 'I-LOC', 'B-LOC', 'I-PER', 'B-PER', 'B-ORG', 'I-ORG'}


## conll2003-SK-NER
ju-bezdek/conll2003-SK-NER

In [20]:
conll2003_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6,
    "B-MISC": 7,
    "I-MISC": 8
}

conll2003 = ner.ReadNERData()
conll2003_words, conll2003_labels = wikiann.read_dataset(
    'ju-bezdek/conll2003-SK-NER',
    conll2003_label_map,
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test Split


  0%|          | 0/3453 [00:00<?, ?it/s]

In [22]:
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC
dataset_label_alignment = {
    'I-MISC': 'O',
    'B-MISC': 'O'
}

In [23]:
# Align the dataset labels to the standard labels
conll2003_labels = ner.align_dataset(conll2003_labels, dataset_label_alignment)

# Evaluate model

In [9]:
alignment = {
'B-LOC': 'B-LOC',
'B-MISC': 'O',
'B-ORG': 'B-ORG',
'I-LOC': 'I-LOC',
'I-MISC': 'O',
'I-ORG': 'I-ORG',
'I-PER': 'I-PER',
'O': 'O'
}

model_name = "xlm-roberta-large-finetuned-conll03-english"
model_name_output = 'xlm-roberta-large'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

config.json:   0%|          | 0.00/852 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the model checkpoint at xlm-roberta-large-finetuned-conll03-english were not used when initializing XLMRobertaForTokenClassification: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
- This IS expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [10]:
print(model_evaluation.model.config.id2label)
print(ner.check_labels(wikiann_labels))

{0: 'B-LOC', 1: 'B-MISC', 2: 'B-ORG', 3: 'I-LOC', 4: 'I-MISC', 5: 'I-ORG', 6: 'I-PER', 7: 'O'}
{'O', 'I-LOC', 'B-LOC', 'I-PER', 'B-PER', 'B-ORG', 'I-ORG'}


### wikiann

In [16]:
data_name = "wikiann"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_words, wikiann_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classif

In [17]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.5701,0.6247,0.5962,4906
1,ORG,0.6843,0.4561,0.5474,3773
2,PER,0.7759,0.8172,0.7960,4601
3,micro,0.6710,0.6435,0.6570,13280
4,macro,0.6768,0.6327,0.6465,13280
5,weighted,0.6739,0.6435,0.6516,13280


In [18]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.0000,0.0000,0.0000,4906
1,B-ORG,0.0000,0.0000,0.0000,3773
2,B-PER,0.0000,0.0000,0.0000,4601
3,I-LOC,0.2635,0.4475,0.3317,4027
4,I-ORG,0.5208,0.3957,0.4497,6993
5,I-PER,0.4931,0.8293,0.6184,6039
6,O,0.8537,0.9901,0.9169,50267
7,accuracy,0.7363,80606,None,None
8,macro,0.3044,0.3804,0.3310,80606
9,weighted,0.6277,0.7363,0.6737,80606


## Conll2003

In [24]:
data_name = "conll2003"
conll_evaluation_output = model_evaluation.evaluate_model(conll2003_words, conll2003_labels)

  0%|          | 0/216 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classif

In [25]:
conll_seqeval = conll_evaluation_output.get_classification('Seqeval')
conll_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.8472,0.8265,0.8367,1556
1,ORG,0.7856,0.8623,0.8222,1598
2,PER,0.8376,0.8827,0.8596,1543
3,micro,0.8220,0.8571,0.8392,4697
4,macro,0.8235,0.8572,0.8395,4697
5,weighted,0.8231,0.8571,0.8393,4697
